# Going Further with JAX — Graph Neural Networks (Student)

 **Optional self-paced extension — NOT part of the two mandatory 45-minute practicals.**

 **Who is it for?** Physical scientists, ML practitioners, and newcomers. This extension reproduces one idea from the PyTorch lab in **JAX** and explains what changes. No prior JAX experience is assumed.

 **By the end, you can:** work with immutable arrays and PRNG keys; explain `jax.jit`, automatic differentiation and asynchronous GPU execution; write a permutation-invariant message-passing operator; distinguish compilation time from steady-state step time.

 **Installation:** JAX, Equinox and Optax must already be available in the remote Python environment. The notebook runs on CPU or GPU. Avoid installing during the 90-minute session. The numerical example is synthetic and educational, **not** a geophysical accuracy benchmark.

**Hedy, one-terminal Euporie + X11:** From a single SSH/X11 connection to `m-login01`, run `bash scripts/start_lab.sh fno student` (or `gnn student`, `infrasound student`, `ffno_map`, `jax_fno student`, `jax_gnn student`). The launcher submits a Slurm GPU allocation, connects to its compute node with SSH X11, enters a Slurm job step and opens this notebook in Euporie. Figures use TkAgg windows via X11; no web browser or local Jupyter is needed. Student edits are saved as private notebook copies. See `README_ONE_TERMINAL_X11.md`.

**Course timing:** mandatory FNO and GNN labs are 45 minutes each; all extensions are optional. **GPU type:** `HEDY_GPU_PROFILE=full` uses an H100; `mig` requests `1g.20gb` when available. Ensure an active site-authorized reservation.

## 0. Device and numerical habits

 JAX arrays are immutable: use `.at[idx].set(...)` rather than in-place assignment. A `jax.random.PRNGKey` is explicit and is split for reproducible sampling. `jax.jit` traces/compiles functions for a fixed shape and dtype; the first call includes compilation. **On GPU, synchronize** with `.block_until_ready()` before timing.

 Equinox (`eqx.Module`) stores trainable arrays in PyTrees; Optax provides optimizer state and updates. Do not assume that a visible device name means you have a valid Slurm GPU allocation.

---
**Authors:** E. Noëlé¹²³⁴, J.-P. Gatti⁵, L. Lehmann⁵, C. Millet⁵, D. Clouteau³

**Affiliations:**
¹ AMIAD — ² CEA — ³ LMPS (Laboratoire de Mécanique Paris-Saclay) — ⁴ Centre Borelli — ⁵ CEA/DAM

**LETMA 2026 Summer School** · Machine Learning on Geophysical Data


In [ ]:
import os, time
import numpy as np
import jax
import jax.numpy as jnp
import equinox as eqx
import optax
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib', 'tk' if os.getenv('LAB_PLOT_MODE', 'inline').lower() == 'tk' else 'inline')
import matplotlib.pyplot as plt

# Faster X11 preview: lower DISPLAY DPI, preserve high-resolution file exports.

if os.environ.get('LAB_PLOT_MODE', 'inline').lower() == 'tk' and os.environ.get('HEDY_FAST_FIGURES', '1') == '1':
    plt.rcParams.update({
        'figure.dpi': 85,
        'savefig.dpi': 200,
        'path.simplify': True,
        'agg.path.chunksize': 12000,
    })

# Choose a non-interactive backend for profiling; on Hedy X11 use LAB_PLOT_MODE=tk.
print("JAX", jax.__version__, "| devices", jax.devices())
print("Platform:", jax.default_backend(), "| SLURM_JOB_ID:", os.getenv("SLURM_JOB_ID"))
key = jax.random.PRNGKey(42)

In [ ]:
from hedy_euporie.profiler import begin_session
HEDY_SESSION = begin_session('jax_gnn_student', backend='jax')

## 1. Physical problem: travel times on a station network

 Consider a synthetic, homogeneous 2D medium with constant speed $v$. A point source at position $(x_s,z_s)$ emits at a **known** origin time $t_0=0$. Station $i$ at $(x_i,z_i)$ measures $t_i=\|\mathbf r_i-\mathbf r_s\|/v + \epsilon_i$. The inversion is simplified for teaching; real seismic localization requires origin-time, velocity uncertainty, phase picking and 3D Earth structure.

 The graph contains stations (nodes), spatial nearest-neighbor links (directed edges) and observed arrival times (node features). The event prediction must be invariant to station ordering.

In [ ]:
import numpy as np
S=12
station_x=np.linspace(0.05,0.95,S, dtype=np.float32)
station_pos=np.stack([station_x, np.zeros_like(station_x)],axis=-1)
d2=((station_pos[:,None,:]-station_pos[None,:,:])**2).sum(axis=-1)
idx=np.argsort(d2,axis=1)[:,1:4]
receivers=jnp.repeat(jnp.arange(S),3)
senders=jnp.asarray(idx.reshape(-1))
print('Nodes:', S, '| directed edges:',len(senders), '| feature per station: (x,z,pick)')

key, src_key, noise_key = jax.random.split(key,3)
sources=jax.random.uniform(src_key,(96,2),minval=0.0,maxval=1.0)
sources=sources.at[:,0].set(0.15+0.70*sources[:,0])
sources=sources.at[:,1].set(0.15+0.55*sources[:,1])
positions=jnp.asarray(station_pos)
travel=jnp.linalg.norm(positions[None,:,:]-sources[:,None,:],axis=-1)
noise=0.003*jax.random.normal(noise_key,travel.shape)
features=jnp.concatenate([jnp.broadcast_to(positions,(96,S,2)),(travel+noise)[...,None]],axis=-1)
train_x, test_x=features[:80],features[80:]
train_y, test_y=sources[:80],sources[80:]
fig,ax=plt.subplots(figsize=(7,3));ax.scatter(station_pos[:,0],station_pos[:,1],marker='v',label='Stations');ax.scatter(np.asarray(sources[:15,0]),np.asarray(sources[:15,1]),marker='*',label='Sources');ax.set(xlabel='Horizontal position',ylabel='Depth (downwards)',ylim=(.85,-.05));ax.legend();ax.grid(alpha=.3);plt.show()

## 2. Exercise JG1 — aggregate messages

 Every directed edge `sender -> receiver` transmits a message. Aggregate by **sum** over incoming edges with functional `.at[receivers].add(messages)`; the aggregate has shape `(stations, hidden)`.

 Unlike PyTorch Geometric, this exercise uses only core JAX arrays. The aggregation is permutation equivariant if node features and edge indices are permuted consistently.

In [ ]:
def reference_aggregate(messages, receivers, n_nodes):
    return jnp.zeros((n_nodes, messages.shape[-1]), dtype=messages.dtype).at[receivers].add(messages)

def aggregate(messages, receivers, n_nodes):
    # TODO JG1: allocate a (n_nodes, hidden) JAX array and add messages
    # to their receiver nodes. Replace this safe reference fallback.
    return reference_aggregate(messages, receivers, n_nodes)

m=jnp.arange(len(senders)*4,dtype=jnp.float32).reshape(-1,4)
np.testing.assert_allclose(np.asarray(aggregate(m,receivers,S)),np.asarray(reference_aggregate(m,receivers,S)))
print('JG1 numerical check: PASS')

## 3. Exercise JG2 — graph-to-event readout

 A single event has many station embeddings but only one source position. Following message passing, take the mean over stations to make a fixed-size, permutation-invariant vector. Which information might mean pooling discard? How could physics-informed features help?

In [ ]:
def reference_pool(node_embeddings):
    return jnp.mean(node_embeddings, axis=0)

def pool_event(node_embeddings):
    # TODO JG2: return (hidden,), not (stations, hidden).
    return reference_pool(node_embeddings)  # safe reference fallback

assert pool_event(jnp.ones((S,8))).shape==(8,)
print('JG2 shape check: PASS')

## 4. Equinox message-passing model

 Equinox modules hold MLP weights as PyTrees. `jax.vmap` applies a shared MLP to all edges and all stations; `eqx.filter_jit` compiles the update. Node embeddings are updated by concatenating their features with aggregated messages. Global pooling produces one prediction per event.

 **Note:** This is not a direct reproduction of the PyTorch GNN benchmark. The data generation, architecture and task difficulty are deliberately simplified.

In [ ]:
class EventGNN(eqx.Module):
    message_net: eqx.nn.MLP
    update_net: eqx.nn.MLP
    readout: eqx.nn.MLP
    hidden: int = eqx.field(static=True)

    def __init__(self, key, hidden=12):
        k1,k2,k3=jax.random.split(key,3)
        self.hidden=hidden
        self.message_net=eqx.nn.MLP(6,hidden,hidden,depth=1,key=k1)
        self.update_net=eqx.nn.MLP(3+hidden,hidden,hidden,depth=1,key=k2)
        self.readout=eqx.nn.MLP(hidden,2,hidden,depth=1,key=k3)

    def __call__(self, nodes, senders, receivers):
        edge_features=jnp.concatenate([nodes[senders],nodes[receivers]],axis=-1)
        messages=jax.vmap(self.message_net)(edge_features)
        combined=aggregate(messages,receivers,nodes.shape[0])
        updated=jax.vmap(self.update_net)(jnp.concatenate([nodes,combined],axis=-1))
        return self.readout(pool_event(updated))

model=EventGNN(key)
optimizer=optax.adam(0.02)
opt_state=optimizer.init(eqx.filter(model,eqx.is_inexact_array))

def loss_fn(model,x,y):
    predict=lambda nodes: model(nodes,senders,receivers)
    return jnp.mean((jax.vmap(predict)(x)-y)**2)

@eqx.filter_jit
def step(model,opt_state,x,y):
    value,grad=eqx.filter_value_and_grad(loss_fn)(model,x,y)
    updates,opt_state=optimizer.update(grad,opt_state,eqx.filter(model,eqx.is_inexact_array))
    return eqx.apply_updates(model,updates),opt_state,value

t0=time.perf_counter()
model,opt_state,value=step(model,opt_state,train_x,train_y)
value.block_until_ready()
print(f'First JIT call (compilation included): {time.perf_counter()-t0:.3f}s')
for i in range(90):
    model,opt_state,value=step(model,opt_state,train_x,train_y)
value.block_until_ready()
print('Training MSE:',float(value),'Test MSE:',float(loss_fn(model,test_x,test_y)))
pred=jax.vmap(lambda n: model(n,senders,receivers))(test_x)
fig,ax=plt.subplots(figsize=(6,4));ax.scatter(np.asarray(test_y[:,0]),np.asarray(pred[:,0]));ax.plot([0,1],[0,1],'--',c='gray');ax.set(xlabel='True x (normalized)',ylabel='Estimated x (normalized)');ax.grid(alpha=.3);plt.show()

## 5. Performance and limitations

 The first JIT call includes compilation, unlike subsequent updates. JAX execution is asynchronous on GPUs: synchronize before measuring. Check also whether a model trained for one station count can be evaluated on other counts without recompilation. This example assumes known origin time, constant speed, fixed node count and station geometry; consequently, its predictive error should not be used to estimate real hypocentre accuracy.

 **Takeaways:** graph aggregation is a scatter sum, pooling removes node order, training uses explicit immutable optimizer state, and compiler latency is a resource cost.

### Optional — save this interactive session measurement

Run the next cell after finishing your exercise. It writes a JSON report with elapsed **interactive** time (including time spent reading and editing), maximum Python-kernel RSS, and (for PyTorch) peak CUDA memory. It does **not** measure Slurm billed time or prove that all cells were executed.

In [ ]:
HEDY_SESSION.report()

## References

- Z. Li et al. *Fourier Neural Operator for Parametric PDEs.* ICLR 2021.
- Z. Li et al. *Neural Operator: Learning Maps Between Function Spaces.* Nature MI **6**, 666 (2024).
- A. Paszke et al. *PyTorch.* NeurIPS 2019.
- J. Bradbury et al. *JAX: Composable Transformations of Python+NumPy Programs* (2018).
- A. Paszke et al. *Equinox: Your Friend for JAX* (2022).
- J. O. Richardson et al. *Optax: Composable Gradient Processing and Optimization* (2023).
